# Fully-Observed Analysis

Analysis of the fully-observed scaling factor recovery experiment.
The network is given teacher spike data and must recover 6 synaptic scaling factors
that were applied as perturbations to the weights.

**Training phases:**
- **CMA-ES**: gradient-free evolutionary search for initialisation
- **Gradient**: Adam optimisation of log-scaling factors with Van Rossum loss

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

from connectome_snns.utils.reproducibility import load_experiment_config
from connectome_snns.visualization import (
    use_project_style,
    CMA_PHASE_COLOR,
    CMA_LOSS_COLOR,
    GRADIENT_LOSS_COLOR,
)
from connectome_snns.visualization.scaling_factors import SF_PATHWAYS

use_project_style()

In [ ]:
RESULTS_DIR = load_experiment_config("experiment.toml")["output_dir"]

df = pd.read_csv(RESULTS_DIR / "training_metrics.csv")
df = df.sort_values("epoch").reset_index(drop=True)

n_cma = (df["epoch"] < 0).sum()
n_grad = (df["epoch"] >= 0).sum()
print(f"CMA-ES rows: {n_cma},  Gradient rows: {n_grad}")
print(f"Epoch range: {df['epoch'].min()} → {df['epoch'].max()}")

In [ ]:
# ── Scaling factor trajectory plot ──────────────────────────────────────────

steps = np.arange(len(df))
cma_mask = df["epoch"] < 0
grad_mask = df["epoch"] >= 0
epoch0_step = cma_mask.sum()  # row index of epoch 0

colors = plt.cm.tab10(np.linspace(0, 0.6, len(SF_PATHWAYS)))

fig, ax = plt.subplots(figsize=(12, 5))

# Shade CMA-ES phase up to and including epoch 0
ax.axvspan(-0.5, epoch0_step + 0.5, color=CMA_PHASE_COLOR, zorder=0)

# Target line
ax.axhline(1.0, color="black", linewidth=1.2, linestyle=":", zorder=2, label="Target")

# Sigma = 2 in epoch/50 units; each gradient step = 5 epochs → sigma = 2*(50/5) = 20 steps
sigma_steps = 20

for (key, label), color in zip(SF_PATHWAYS, colors):
    values = np.full(len(df), np.nan)
    cma_col = f"cma_es_scaling_factors/{key}_value"
    grad_col = f"scaling_factors/{key}_value"
    if cma_col in df.columns:
        values[cma_mask.values] = df.loc[cma_mask, cma_col].values
    if grad_col in df.columns:
        grad_vals = df.loc[grad_mask, grad_col].values.astype(float)
        values[grad_mask.values] = gaussian_filter1d(grad_vals, sigma=sigma_steps)
    ax.plot(steps, values, color=color, linewidth=1.8, label=label, zorder=3)

# x-ticks: gradient steps only, labelled as epoch/50
grad_steps = df.index[grad_mask].tolist()
grad_epoch = df.loc[grad_mask, "epoch"].tolist()
tick_indices = np.linspace(0, len(grad_steps) - 1, 5, dtype=int)
grad_ticks = [grad_steps[i] for i in tick_indices]
grad_labels = [str(int(grad_epoch[i] / 50)) for i in tick_indices]

ax.set_xticks(grad_ticks)
ax.set_xticklabels(grad_labels)
ax.set_xlim(-0.5, len(df) - 0.5)
ax.set_ylim(bottom=0)

ax.set_ylabel("Scaling Factor")
ax.set_xlabel("Epoch")
ax.set_title("Scaling Factor Trajectories")
ax.legend(loc="upper right", frameon=True, ncol=2)

plt.tight_layout()
plt.show()

# Print final values vs targets
final = df[grad_mask].iloc[-1]
print("\nFinal scaling factors:")
print(f"  {'Connection':<8} {'Value':>8}  {'Target':>8}  {'Error':>8}")
for key, label in SF_PATHWAYS:
    v = final.get(f"scaling_factors/{key}_value", np.nan)
    t = final.get(f"scaling_factors/{key}_target", 1.0)
    print(f"  {label:<8} {v:>8.4f}  {t:>8.4f}  {abs(v - t):>8.4f}")

In [ ]:
# ── Loss curves ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# CMA-ES total loss
cma_df = df[cma_mask].copy()
axes[0].plot(
    np.arange(len(cma_df)),
    cma_df["cma_es_loss/total"],
    color=CMA_LOSS_COLOR,
    linewidth=1.8,
)
axes[0].set_title("CMA-ES Total Loss")
axes[0].set_xlabel("CMA-ES Generation")
axes[0].set_ylabel("Loss")

# Gradient Van Rossum loss
grad_df = df[grad_mask].copy()
axes[1].plot(
    grad_df["epoch"],
    grad_df["van_rossum_loss"],
    color=GRADIENT_LOSS_COLOR,
    linewidth=1.5,
)
axes[1].set_title("Van Rossum Loss (Gradient Phase)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")

plt.tight_layout()
plt.show()